# SO-101 エサやり — SmolVLA 微調整（Colab用）

ランタイム → ランタイムのタイプを変更 → **GPU（T4以上）** を選んでから上から順に実行してください。

事前に `feed_smolvla.zip` を Google ドライブの `マイドライブ/physical_ai/` に置いておきます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/work && cd /content/work && unzip -q -o /content/drive/MyDrive/physical_ai/feed_smolvla.zip
%cd /content/work/feed_smolvla
!ls

In [ ]:
# 依存パッケージ（数分かかります）
# smolvla: モデル本体 / dataset: データ読み込み(av等) / training: 学習(accelerate等)
!pip install -q "lerobot[smolvla,dataset,training]" mujoco imageio imageio-ffmpeg
!python -c "import av, accelerate, lerobot; print('lerobot', lerobot.__version__, '| av', av.__version__, '| accelerate', accelerate.__version__)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# MuJoCoをGPU(EGL)で描画する設定
import os
os.environ['MUJOCO_GL'] = 'egl'
%env MUJOCO_GL=egl
!python -c "from feed_env import FeedEnv; e=FeedEnv(); o=e.reset(0); print('sim ok', o['top'].shape)"

## 1. 学習
T4で10,000ステップおよそ2〜3時間が目安です。途中のチェックポイントは5,000ステップごとに保存されます。

In [ ]:
!STEPS=10000 BATCH=16 bash train_smolvla.sh

## 2. シミュレーションで評価（学習に使っていない30シード）

In [ ]:
!python eval_smolvla.py --ckpt outputs/smolvla_feed/checkpoints/last/pretrained_model --n 30 --video videos/rollout.mp4

## 3. 言葉の言い換えを試す
学習時のタスク文は "feed the pet" だけです。言い換えや無関係な指示でどう振る舞うかを見ると、言語条件付けの効き具合がわかります。

In [ ]:
CK='outputs/smolvla_feed/checkpoints/last/pretrained_model'
!python eval_smolvla.py --ckpt $CK --n 10 --task "give the dog its food"
!python eval_smolvla.py --ckpt $CK --n 10 --task "pick up the red cup"

In [ ]:
from IPython.display import Video
import glob
for f in sorted(glob.glob('videos/*.mp4'))[:2]:
    display(Video(f, embed=True, width=480))

## 4. 結果をドライブに保存

In [ ]:
!mkdir -p /content/drive/MyDrive/physical_ai/smolvla_results
!cp -r outputs/smolvla_feed/checkpoints/last/pretrained_model videos /content/drive/MyDrive/physical_ai/smolvla_results/
!ls /content/drive/MyDrive/physical_ai/smolvla_results